# ЛР1 и ЛР2 — демонстрация в Google Colab
**Перед запуском:** Runtime → Change runtime type → **T4 GPU**. Затем Runtime → Run all.
Это демонстрация на GPU Colab (не на учебном кластере): показывает, что код собирается и работает.

In [ ]:
!nvidia-smi
!nvcc --version

## 1. Получаем код (после push в GitHub)

In [ ]:
import os
if not os.path.exists('GPU-Programming-Labs'):
    !git clone https://github.com/NBx03/GPU-Programming-Labs.git
%cd GPU-Programming-Labs

Если репозиторий приватный или ещё не запушен — вместо клонирования загрузите zip с папками `common`, `lab1`, `lab2`:
```python
from google.colab import files; files.upload()      # выберите gpu_labs.zip
!unzip -o gpu_labs.zip -d GPU-Programming-Labs
%cd GPU-Programming-Labs
```

In [ ]:
import subprocess
cc = subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split()[0]
ARCH = 'sm_' + cc.replace('.', '')
print('compute capability', cc, '->', ARCH)
!apt-get -qq install -y libpng-dev > /dev/null

## 2. ЛР1: sin / sinf / __sinf (быстрый прогон на 10⁸ элементов)

In [ ]:
%cd lab1
!make ARCH={ARCH}
!mkdir -p results
!./sin_precision --n 100000000 --reps 3

Полный прогон N = 10⁹ (чанками по 2²⁸ элементов; ~2–5 минут):

In [ ]:
!./sin_precision --reps 3

Сборка с `-use_fast_math`: `sinf` должен превратиться в `__sinf` (сравните err и время):

In [ ]:
!./sin_precision_fastmath --n 100000000 --reps 3 --no-micro
%cd ..

## 3. ЛР2: фильтры изображений

In [ ]:
%cd lab2
!make ARCH={ARCH}
!mkdir -p examples results
!./filters --gen 1024 examples/input.png
!./filters examples/input.png examples/out_blur.png   blur --radius 3
!./filters examples/input.png examples/out_sobel.png  sobel
!./filters examples/input.png examples/out_median.png median

В колонке `differing bytes / max diff` должно быть `0 / 0` — результат GPU совпадает с CPU-эталоном.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
names = ['input', 'out_blur', 'out_sobel', 'out_median']
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, n in zip(ax, names):
    a.imshow(Image.open(f'examples/{n}.png')); a.set_title(n); a.axis('off')
plt.show()

Бенчмарк накладных расходов (для 4096 добавьте в `--sizes`, но CPU-эталон на 2 ядрах Colab будет считать долго):

In [ ]:
!./filters --bench --sizes 512,2048 --radii 1,4,7 --reps 3 --csv results/bench_colab.csv
%cd ..